# Proteus Arc — EEG-Based Alzheimer's Disease Classification
### Conventional Machine Learning Pipeline (SVM, Random Forest, Logistic Regression)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook implements a complete, participant-independent EEG classification pipeline:

1. Install and import libraries
2. Define EEG frequency bands
3. Extract time-domain, Hjorth and band-power features per epoch
4. Load and validate your dataset
5. Explore class distribution
6. Build the feature matrix
7. Split data **by participant** (no leakage)
8. Train three models: Logistic Regression, SVM (RBF), Random Forest
9. Evaluate (accuracy, balanced accuracy, macro P/R/F1, ROC-AUC, confusion matrices)
10. Compare models, inspect Random Forest feature importance
11. Save / load the trained model

### Required inputs for real experiments
| Variable | Description |
|---|---|
| `X_epochs` | EEG array, shape `(n_epochs, n_channels, n_samples)` |
| `y` | Diagnostic label per epoch |
| `participant_ids` | Participant identifier per epoch |
| `sfreq` | Sampling frequency in Hz (must match your recording) |
| `channel_names` | List of EEG channel names |

> **Important:** By default this notebook runs on a small **synthetic demo dataset** so you can verify the code works end to end.
> Results from the demo are **meaningless scientifically**. Set `USE_SYNTHETIC_DEMO = False` in Cell 5 and load your real data for actual results.

## Cell 1 — Install libraries

In [ ]:
!pip install -q numpy scipy pandas scikit-learn matplotlib seaborn joblib

## Cell 2 — Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from scipy.signal import welch
from scipy.stats import skew, kurtosis

from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix, roc_auc_score
)

import warnings
warnings.filterwarnings("ignore")

# np.trapezoid exists in NumPy >= 2.0; fall back to np.trapz on older versions
trapezoid = getattr(np, "trapezoid", None) or np.trapz

RANDOM_STATE = 42
print("All libraries imported successfully!")

## Cell 3 — Define EEG frequency bands
Standard clinical bands used for spectral power features. Gamma is excluded.

In [ ]:
BANDS = {
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
}

print("EEG frequency bands:")
for band, (low, high) in BANDS.items():
    print(f"{band.capitalize()}: {low}-{high} Hz")

## Cell 4 — EEG feature extraction
For every channel of every epoch, the following features are computed:

* **Time-domain:** mean, std, variance, RMS, min, max, range, skewness, kurtosis
* **Hjorth parameters:** activity, mobility, complexity
* **Frequency-domain (Welch PSD):** absolute and relative power in delta, theta, alpha, beta

In [ ]:
def extract_epoch_features(epoch, sfreq, channel_names=None):
    """
    Extract time-domain and frequency-domain features from one epoch.

    Parameters
    ----------
    epoch : numpy array, shape (channels, samples)
    sfreq : float, sampling frequency in Hz
    channel_names : list, optional

    Returns
    -------
    dict of extracted features
    """
    epoch = np.asarray(epoch, dtype=float)

    if epoch.ndim != 2:
        raise ValueError("Each epoch must have shape (channels, samples).")

    n_channels, n_samples = epoch.shape

    if sfreq <= 0 or n_samples < 8:
        raise ValueError("Invalid sampling frequency or insufficient samples.")

    if not np.isfinite(epoch).all():
        raise ValueError("EEG epoch contains NaN or infinite values.")

    if channel_names is None:
        channel_names = [f"ch_{i}" for i in range(n_channels)]

    if len(channel_names) != n_channels:
        raise ValueError("Channel names do not match the number of channels.")

    features = {}

    for ch_idx, ch_name in enumerate(channel_names):
        signal = epoch[ch_idx]
        prefix = str(ch_name)

        # ---- Time-domain features ----
        features[f"{prefix}_mean"] = np.mean(signal)
        features[f"{prefix}_std"] = np.std(signal)
        features[f"{prefix}_variance"] = np.var(signal)
        features[f"{prefix}_rms"] = np.sqrt(np.mean(signal ** 2))
        features[f"{prefix}_minimum"] = np.min(signal)
        features[f"{prefix}_maximum"] = np.max(signal)
        features[f"{prefix}_range"] = np.max(signal) - np.min(signal)
        features[f"{prefix}_skewness"] = skew(signal, bias=False)
        features[f"{prefix}_kurtosis"] = kurtosis(signal, bias=False)

        # ---- Hjorth parameters ----
        activity = np.var(signal)
        first_derivative = np.diff(signal)
        second_derivative = np.diff(first_derivative)
        first_var = np.var(first_derivative)
        second_var = np.var(second_derivative)

        mobility = np.sqrt(first_var / activity) if activity > 0 else 0.0
        complexity = (
            np.sqrt(second_var / first_var) / mobility
            if first_var > 0 and mobility > 0 else 0.0
        )

        features[f"{prefix}_hjorth_activity"] = activity
        features[f"{prefix}_hjorth_mobility"] = mobility
        features[f"{prefix}_hjorth_complexity"] = complexity

        # ---- Frequency-domain features (Welch PSD) ----
        freqs, psd = welch(
            signal, fs=sfreq,
            nperseg=min(n_samples, max(8, int(sfreq * 2)))
        )

        total_mask = (freqs >= 0.5) & (freqs <= min(30.0, sfreq / 2))
        total_power = (
            trapezoid(psd[total_mask], freqs[total_mask])
            if total_mask.sum() >= 2 else 0.0
        )

        for band_name, (low, high) in BANDS.items():
            effective_high = min(high, sfreq / 2)
            mask = (freqs >= low) & (freqs < effective_high)
            power = trapezoid(psd[mask], freqs[mask]) if mask.sum() >= 2 else 0.0

            features[f"{prefix}_{band_name}_power"] = power
            features[f"{prefix}_{band_name}_relative_power"] = (
                power / total_power if total_power > 0 else 0.0
            )

    return features


def build_feature_table(X_epochs, sfreq, channel_names=None):
    """X_epochs shape: (epochs, channels, samples). Returns a DataFrame."""
    rows = []
    for i, epoch in enumerate(X_epochs):
        rows.append(extract_epoch_features(epoch, sfreq, channel_names))
        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{len(X_epochs)} epochs")
    return pd.DataFrame(rows)


print("EEG feature extraction functions are ready.")

## Cell 5 — Load your EEG dataset
Your dataset must provide `X_epochs`, `y`, `participant_ids`, `sfreq`, and `channel_names`
(expected shape: `(number_of_epochs, number_of_channels, number_of_samples)`).

* **`USE_SYNTHETIC_DEMO = True`** → generates a small fake dataset so the whole notebook runs. *Not real results.*
* **`USE_SYNTHETIC_DEMO = False`** → fill in the `# LOAD YOUR REAL DATA` block below.

In [ ]:
USE_SYNTHETIC_DEMO = True   # <-- set to False when using your real dataset

if USE_SYNTHETIC_DEMO:
    # ---------- SYNTHETIC DEMO DATA (for testing the pipeline only) ----------
    rng = np.random.default_rng(RANDOM_STATE)
    sfreq = 256
    channel_names = ["Fp1", "Fp2", "F3", "F4"]
    classes = ["Control", "MCI", "AD"]
    n_participants_per_class, epochs_per_participant = 20, 10
    n_samples = sfreq * 4
    t = np.arange(n_samples) / sfreq

    X_list, y_list, pid_list = [], [], []
    for c_idx, cls in enumerate(classes):
        for p in range(n_participants_per_class):
            pid = f"{cls}_{p:02d}"
            for _ in range(epochs_per_participant):
                epoch = []
                for _ch in channel_names:
                    alpha_amp = 2.0 - 0.6 * c_idx   # alpha drops with severity
                    theta_amp = 1.0 + 0.6 * c_idx   # theta rises with severity
                    sig = (alpha_amp * np.sin(2*np.pi*10*t + rng.uniform(0, 6.28))
                           + theta_amp * np.sin(2*np.pi*6*t + rng.uniform(0, 6.28))
                           + rng.normal(0, 1.5, n_samples))
                    epoch.append(sig)
                X_list.append(epoch); y_list.append(cls); pid_list.append(pid)

    X_epochs = np.array(X_list)
    y = np.array(y_list)
    participant_ids = np.array(pid_list)
    print("Using SYNTHETIC demo data. Results are NOT scientifically meaningful.")

else:
    # ---------- LOAD YOUR REAL DATA ----------
    # X_epochs = ...          # shape (epochs, channels, samples)
    # y = ...                 # diagnostic label per epoch
    # participant_ids = ...   # participant ID per epoch
    # sfreq = ...             # e.g. 256  (use your dataset's real rate!)
    # channel_names = [...]   # e.g. ["Fp1", "Fp2", ...]
    raise NotImplementedError("Load your real EEG data in this block.")


def validate_eeg_data(X_epochs, y, participant_ids, sfreq):
    X_epochs = np.asarray(X_epochs)
    y = np.asarray(y)
    participant_ids = np.asarray(participant_ids)

    if X_epochs.ndim != 3:
        raise ValueError("EEG data must have 3 dimensions: (epochs, channels, samples).")
    if not (len(X_epochs) == len(y) == len(participant_ids)):
        raise ValueError("EEG epochs, labels, and participant IDs must have equal lengths.")
    if len(X_epochs) == 0:
        raise ValueError("The dataset is empty.")
    if sfreq <= 0:
        raise ValueError("Sampling frequency must be positive.")
    if not np.isfinite(X_epochs).all():
        raise ValueError("EEG data contains missing or infinite values.")
    if pd.isna(y).any():
        raise ValueError("Diagnostic labels contain missing values.")
    if pd.isna(participant_ids).any():
        raise ValueError("Participant IDs contain missing values.")

    print("Dataset validation successful.")
    print("Number of epochs:", len(X_epochs))
    print("Number of channels:", X_epochs.shape[1])
    print("Samples per epoch:", X_epochs.shape[2])
    print("Sampling frequency:", sfreq, "Hz")
    print("Unique participants:", len(np.unique(participant_ids)))
    print("Classes:", np.unique(y))


validate_eeg_data(X_epochs, y, participant_ids, sfreq)

## Cell 6 — Explore the dataset

In [ ]:
def explore_dataset(X_epochs, y, participant_ids):
    y = np.asarray(y)
    participant_ids = np.asarray(participant_ids)

    print("Dataset shape:", X_epochs.shape)
    print("Number of epochs:", len(y))
    print("Number of participants:", len(np.unique(participant_ids)))
    print("\nClass distribution:")

    class_counts = pd.Series(y).value_counts()
    print(class_counts)

    plt.figure(figsize=(9, 5))
    class_counts.plot(kind="bar", edgecolor="black")
    plt.title("EEG Diagnostic Class Distribution")
    plt.xlabel("Diagnostic Class")
    plt.ylabel("Number of Epochs")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


explore_dataset(X_epochs, y, participant_ids)

## Cell 7 — Extract the feature matrix
Converts each epoch into a vector of features. Features are also saved to CSV.

In [ ]:
features_df = build_feature_table(X_epochs, sfreq=sfreq, channel_names=channel_names)

print("Feature matrix shape:", features_df.shape)
display(features_df.head())

features_df.to_csv("eeg_extracted_features.csv", index=False)

# Download from Colab if required:
# from google.colab import files
# files.download("eeg_extracted_features.csv")

## Cell 8 — Split data by participant
`GroupShuffleSplit` guarantees epochs from the same participant never appear in both train and test sets,
preventing data leakage and over-optimistic results.

In [ ]:
def split_by_participant(X, y, participant_ids, test_size=0.20):
    X = np.asarray(X)
    y = np.asarray(y)
    groups = np.asarray(participant_ids)

    if not (len(X) == len(y) == len(groups)):
        raise ValueError("X, y, and participant IDs must have equal lengths.")

    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    groups_train, groups_test = groups[train_idx], groups[test_idx]

    assert set(groups_train).isdisjoint(set(groups_test)), "Participant leakage detected!"

    print("Training epochs:", len(train_idx))
    print("Testing epochs:", len(test_idx))
    print("Training participants:", len(np.unique(groups_train)))
    print("Testing participants:", len(np.unique(groups_test)))
    print("Training classes:", np.unique(y_train))
    print("Testing classes:", np.unique(y_test))

    return X_train, X_test, y_train, y_test, groups_train, groups_test


X = features_df.to_numpy(dtype=float)

X_train, X_test, y_train, y_test, groups_train, groups_test = split_by_participant(
    X, y, participant_ids
)

## Cell 9 — Define the three ML models
* **Logistic Regression** – linear baseline (median imputation + standardization)
* **SVM (RBF kernel)** – non-linear classifier (median imputation + standardization)
* **Random Forest** – 300 trees; no scaling needed

All use `class_weight="balanced"` to handle class imbalance.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "SVM": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            kernel="rbf", C=1.0, gamma="scale", class_weight="balanced",
            probability=True, random_state=RANDOM_STATE)),
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=300, min_samples_leaf=2, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
}

print("Models defined:")
for model_name in models:
    print("-", model_name)

## Cell 10 — Train and evaluate models
Reports accuracy, balanced accuracy, macro precision/recall/F1, ROC-AUC (binary or one-vs-rest macro),
a classification report and a confusion matrix per model.

In [ ]:
def train_and_evaluate_models(models, X_train, X_test, y_train, y_test):
    results = []
    trained_models = {}

    for model_name, model in models.items():
        print("\n" + "=" * 60)
        print("MODEL:", model_name)
        print("=" * 60)

        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        trained_models[model_name] = model

        metrics = {
            "Model": model_name,
            "Accuracy": accuracy_score(y_test, predictions),
            "Balanced Accuracy": balanced_accuracy_score(y_test, predictions),
            "Macro Precision": precision_score(y_test, predictions, average="macro", zero_division=0),
            "Macro Recall": recall_score(y_test, predictions, average="macro", zero_division=0),
            "Macro F1": f1_score(y_test, predictions, average="macro", zero_division=0),
        }

        # ROC-AUC when supported
        try:
            probabilities = model.predict_proba(X_test)
            if len(model.classes_) == 2:
                positive_class = model.classes_[1]
                binary_targets = (np.asarray(y_test) == positive_class).astype(int)
                metrics["ROC-AUC"] = roc_auc_score(binary_targets, probabilities[:, 1])
            else:
                metrics["ROC-AUC"] = roc_auc_score(
                    y_test, probabilities, labels=model.classes_,
                    multi_class="ovr", average="macro")
        except (ValueError, AttributeError):
            metrics["ROC-AUC"] = np.nan

        results.append(metrics)

        print("\nClassification Report:")
        print(classification_report(y_test, predictions, zero_division=0))

        cm = confusion_matrix(y_test, predictions, labels=model.classes_)
        plt.figure(figsize=(7, 5))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=model.classes_, yticklabels=model.classes_)
        plt.title(f"{model_name} Confusion Matrix")
        plt.xlabel("Predicted Label")
        plt.ylabel("True Label")
        plt.tight_layout()
        plt.show()

    results_df = pd.DataFrame(results).sort_values("Macro F1", ascending=False)
    return results_df, trained_models


results_df, trained_models = train_and_evaluate_models(
    models, X_train, X_test, y_train, y_test
)
display(results_df)

## Cell 11 — Visualize model performance

In [ ]:
def plot_model_comparison(results_df):
    metrics = ["Accuracy", "Balanced Accuracy", "Macro Precision", "Macro Recall", "Macro F1"]
    available = [m for m in metrics if m in results_df.columns]
    plot_df = results_df.set_index("Model")[available]

    ax = plot_df.plot(kind="bar", figsize=(12, 6))
    ax.set_title("EEG Classification Model Comparison")
    ax.set_ylabel("Score")
    ax.set_xlabel("Machine Learning Model")
    ax.set_ylim(0, 1.05)
    plt.xticks(rotation=0)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


plot_model_comparison(results_df)

## Cell 12 — Random Forest feature importance
Shows which EEG features (channel + feature type) contribute most to the Random Forest's decisions.

In [ ]:
def analyze_feature_importance(trained_models, feature_names, model_name="Random Forest", top_n=20):
    if model_name not in trained_models:
        raise ValueError(f"{model_name} has not been trained.")

    classifier = trained_models[model_name].named_steps["classifier"]

    if not hasattr(classifier, "feature_importances_"):
        raise ValueError("Selected model does not support built-in feature importance.")

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": classifier.feature_importances_,
    }).sort_values("Importance", ascending=False).head(top_n)

    display(importance_df)

    plt.figure(figsize=(10, 7))
    sns.barplot(data=importance_df, x="Importance", y="Feature")
    plt.title(f"Top {top_n} Features - {model_name}")
    plt.tight_layout()
    plt.show()

    return importance_df


importance_df = analyze_feature_importance(
    trained_models, features_df.columns.tolist(), model_name="Random Forest", top_n=20
)

## Cell 13 — Save the trained model
Saves a full pipeline (imputer + scaler + classifier) with `joblib`.

In [ ]:
import joblib

def save_trained_model(trained_models, model_name, filename="eeg_alzheimer_classifier.joblib"):
    if model_name not in trained_models:
        raise ValueError(f"Model '{model_name}' is not available.")

    joblib.dump(trained_models[model_name], filename)
    print("Model saved successfully:", filename)
    print("File size:", round(os.path.getsize(filename) / 1024, 2), "KB")
    return filename


save_trained_model(trained_models, "Random Forest")

# To download from Colab:
# from google.colab import files
# files.download("eeg_alzheimer_classifier.joblib")

## Cell 14 — Load the saved model

In [ ]:
def load_trained_model(filename="eeg_alzheimer_classifier.joblib"):
    model = joblib.load(filename)
    print("Model loaded successfully.")
    return model


loaded_model = load_trained_model()
predictions = loaded_model.predict(X_test)
print("Predictions:", predictions[:10])

## Notes & limitations
* Preprocessing (filtering, artifact removal, epoching) is **not** included; apply it before this pipeline.
* Use the real sampling rate of your dataset — band powers depend on it.
* With small numbers of participants, a single split is high-variance. `StratifiedGroupKFold` and `cross_val_score` are imported so you can add participant-grouped cross-validation.
* Confirm which diagnostic classes your dataset actually contains before interpreting results.
* This is a research tool, not a diagnostic device.